<a href="https://colab.research.google.com/github/jiyajain611-sketch/hcl/blob/main/Jiya%20Jain(2026B03110066).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import numpy as np
import pandas as pd

# TASK 1: Load and inspect the data
# ---------------------------------------------------------------------------
print("TASK 1: LOAD AND INSPECT THE DATA")
df = pd.read_csv("/content/data_set_of_air_quality_1000.csv")

print("Rows and columns:", df.shape)
print("\nFirst 5 rows:")
print(df.head())
print("\nColumn types:")
print(df.dtypes)
print("\nQuick statistics:")
print(df.describe())

TASK 1: LOAD AND INSPECT THE DATA
Rows and columns: (1000, 7)

First 5 rows:
         Date        City   PM25   PM10   NO2  Temperature_C  Humidity_Percent
0  01-01-2025       Delhi   28.6  111.6  47.4           15.4                26
1  02-01-2025      Jaipur   59.8  160.7  51.0           30.4                49
2  03-01-2025  Chandigarh  105.0  157.6  40.5           18.4                68
3  04-01-2025  Chandigarh   86.5   55.6  28.9           26.5                33
4  05-01-2025  Chandigarh  134.6  154.1  47.5           51.2                88

Column types:
Date                 object
City                 object
PM25                float64
PM10                float64
NO2                 float64
Temperature_C       float64
Humidity_Percent      int64
dtype: object

Quick statistics:
             PM25         PM10         NO2  Temperature_C  Humidity_Percent
count  976.000000  1000.000000  1000.00000    1000.000000       1000.000000
mean    82.428996   145.487900    35.92660      26.17

In [9]:
# --------------------------------------------------------------------------
# TASK 2: Find missing values
# ---------------------------------------------------------------------------
print("\n\nTASK 2: MISSING VALUES IN EACH COLUMN")
print(df.isnull().sum())



TASK 2: MISSING VALUES IN EACH COLUMN
Date                 0
City                 0
PM25                24
PM10                 0
NO2                  0
Temperature_C        0
Humidity_Percent     0
dtype: int64


In [10]:
# TASK 3: Handle missing PM2.5 values
# ---------------------------------------------------------------------------
print("\n\nTASK 3: FILL MISSING PM2.5 VALUES")

# Keep a copy of the original column so we never lose the real data.
df["PM25_Raw"] = df["PM25"]

# For each city, fill the gaps with that city's median PM2.5.
for city in df["City"].unique():
    city_rows = df["City"] == city                      # rows of this city
    city_median = df.loc[city_rows, "PM25"].median()    # typical value for it
    missing_rows = city_rows & df["PM25"].isnull()      # city rows with a gap
    df.loc[missing_rows, "PM25"] = city_median

print("Missing PM2.5 values left:", df["PM25"].isnull().sum())
print("""
Why the median of each city?
  - The median is not pulled up by a few extreme spikes (the mean is).
  - Using the city's own median keeps the differences between cities.
  - Deleting the rows would waste good PM10, NO2 and temperature data.
""")



TASK 3: FILL MISSING PM2.5 VALUES
Missing PM2.5 values left: 0

Why the median of each city?
  - The median is not pulled up by a few extreme spikes (the mean is).
  - Using the city's own median keeps the differences between cities.
  - Deleting the rows would waste good PM10, NO2 and temperature data.



In [11]:
# TASK 4: Convert Date to datetime and get the month
# ---------------------------------------------------------------------------
print("TASK 4: DATE AND MONTH")
df["Date"] = pd.to_datetime(df["Date"], format="%d-%m-%Y")  # day-month-year
df["Month"] = df["Date"].dt.month                           # 1 to 12
print(df[["Date", "Month"]].head())
print("Date type is now:", df["Date"].dtype)
print("Data runs from", df["Date"].min().date(), "to", df["Date"].max().date())


TASK 4: DATE AND MONTH
        Date  Month
0 2025-01-01      1
1 2025-01-02      1
2 2025-01-03      1
3 2025-01-04      1
4 2025-01-05      1
Date type is now: datetime64[ns]
Data runs from 2025-01-01 to 2027-09-27


In [12]:
# TASK 5: Average, median and maximum PM2.5
# ---------------------------------------------------------------------------
print("\n\nTASK 5: AVERAGE, MEDIAN AND MAXIMUM PM2.5")
average_pm25 = df["PM25"].mean()
median_pm25 = df["PM25"].median()
max_pm25 = df["PM25"].max()
print("Average:", round(average_pm25, 2))
print("Median :", round(median_pm25, 2))
print("Maximum:", round(max_pm25, 2))




TASK 5: AVERAGE, MEDIAN AND MAXIMUM PM2.5
Average: 82.43
Median : 82.15
Maximum: 192.3


In [13]:
# TASK 6: Rank cities by average PM2.5
# ---------------------------------------------------------------------------
print("\n\nTASK 6: CITY RANKING (highest average PM2.5 first)")
city_ranking = df.groupby("City")["PM25"].mean()
city_ranking = city_ranking.sort_values(ascending=False).round(2)
print(city_ranking)

worst_city = city_ranking.index[0]       # first city in the sorted list
worst_city_avg = city_ranking.iloc[0]



TASK 6: CITY RANKING (highest average PM2.5 first)
City
Chandigarh    85.82
Jaipur        82.65
Delhi         81.24
Lucknow       80.05
Name: PM25, dtype: float64


In [14]:
# TASK 7: Dates with the highest PM2.5
# ---------------------------------------------------------------------------
print("\n\nTASK 7: TOP 10 DATES WITH THE HIGHEST PM2.5")
top_dates = df.nlargest(10, "PM25")[["Date", "City", "PM25"]]
print(top_dates.to_string(index=False))

peak_row = df.loc[df["PM25"].idxmax()]   # the single highest reading




TASK 7: TOP 10 DATES WITH THE HIGHEST PM2.5
      Date       City  PM25
2026-07-22      Delhi 192.3
2027-07-02      Delhi 192.3
2025-04-15 Chandigarh 188.6
2026-02-20     Jaipur 186.5
2026-04-24      Delhi 183.9
2026-09-13 Chandigarh 183.9
2025-10-22    Lucknow 179.6
2025-08-09      Delhi 175.1
2025-05-19      Delhi 174.9
2027-06-27      Delhi 174.6


In [15]:
# TASK 8: Compare PM10 and PM2.5 for each city
# ---------------------------------------------------------------------------
print("\n\nTASK 8: AVERAGE PM10 vs PM2.5 BY CITY")
pm_compare = df.groupby("City")[["PM10", "PM25"]].mean().round(2)
pm_compare["PM25_share_of_PM10"] = (pm_compare["PM25"] / pm_compare["PM10"]).round(2)
print(pm_compare)

highest_pm10_city = pm_compare["PM10"].idxmax()



TASK 8: AVERAGE PM10 vs PM2.5 BY CITY
              PM10   PM25  PM25_share_of_PM10
City                                         
Chandigarh  151.18  85.82                0.57
Delhi       146.17  81.24                0.56
Jaipur      141.92  82.65                0.58
Lucknow     142.36  80.05                0.56


In [16]:
# TASK 9: Air-quality category column
# ---------------------------------------------------------------------------
print("\n\nTASK 9: AIR-QUALITY CATEGORIES")
print("Limits used (examples - replace with your instructor's):")
print(CATEGORY_LIMITS)

# pd.cut puts every PM2.5 value into the right group.
df["AQ_Category"] = pd.cut(df["PM25"], bins=CATEGORY_LIMITS,
                           labels=CATEGORY_NAMES, include_lowest=True)
category_counts = df["AQ_Category"].value_counts().reindex(CATEGORY_NAMES)
print(category_counts)

# Percent of readings that are Poor, Very Poor or Severe.
bad_percent = category_counts[["Poor", "Very Poor", "Severe"]].sum() / len(df) * 100




TASK 9: AIR-QUALITY CATEGORIES
Limits used (examples - replace with your instructor's):
[0, 30, 60, 90, 120, 250, 1000]
AQ_Category
Good             78
Satisfactory    176
Moderate        349
Poor            262
Very Poor       135
Severe            0
Name: count, dtype: int64


In [17]:
# TASK 10: Monthly average PM2.5 by city
# ---------------------------------------------------------------------------
print("\n\nTASK 10: MONTHLY AVERAGE PM2.5 BY CITY")
monthly = df.groupby(["City", "Month"])["PM25"].mean()
monthly_table = monthly.unstack(level="City").round(2)   # months as rows
print(monthly_table)
print("(Month 1 = January ... 12 = December. Years are combined.)")

month_average = df.groupby("Month")["PM25"].mean()   # all cities together
highest_month = month_average.idxmax()
spread_percent = (month_average.max() - month_average.min()) / average_pm25 * 100
print("\nMonth with the highest average PM2.5:", highest_month)




TASK 10: MONTHLY AVERAGE PM2.5 BY CITY
City   Chandigarh  Delhi  Jaipur  Lucknow
Month                                    
1           88.99  83.41   77.39    84.31
2           82.08  86.66   77.34    84.69
3           82.50  78.10   96.40    79.10
4           90.45  71.54   85.27    71.52
5           86.44  86.51   77.41    80.60
6           80.54  86.58   81.03    71.76
7           71.42  85.33   82.47    86.23
8           85.45  87.56   70.53    82.72
9           89.94  67.09   85.99    83.09
10          80.47  79.09   83.47    88.39
11          94.19  79.32   86.67    74.02
12         101.78  75.29   90.34    75.87
(Month 1 = January ... 12 = December. Years are combined.)

Month with the highest average PM2.5: 1


In [18]:
# TASK 11: Standard deviation and percentiles with NumPy
# ---------------------------------------------------------------------------
print("\n\nTASK 11: STANDARD DEVIATION AND PERCENTILES (NumPy)")
pm25_values = df["PM25"].to_numpy()
print("Standard deviation:", round(np.std(pm25_values), 2))

for p in [25, 50, 75, 90, 95]:
    print("Percentile", p, ":", round(np.percentile(pm25_values, p), 2))



TASK 11: STANDARD DEVIATION AND PERCENTILES (NumPy)
Standard deviation: 35.2
Percentile 25 : 59.58
Percentile 50 : 82.15
Percentile 75 : 104.72
Percentile 90 : 128.12
Percentile 95 : 141.7


In [19]:
# TASK 12: Outliers using the IQR rule
# ---------------------------------------------------------------------------
print("\n\nTASK 12: OUTLIERS (IQR RULE)")
# Use only the real readings (not the ones we filled in).
real_values = df["PM25_Raw"].dropna().to_numpy()
q1 = np.percentile(real_values, 25)
q3 = np.percentile(real_values, 75)
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
print("Q1:", round(q1, 2), " Q3:", round(q3, 2), " IQR:", round(iqr, 2))
print("A value is an outlier if it is below", round(lower_fence, 2),
      "or above", round(upper_fence, 2))

outliers = df[(df["PM25_Raw"] < lower_fence) | (df["PM25_Raw"] > upper_fence)]
print("Number of outliers:", len(outliers))
print(outliers[["Date", "City", "PM25"]].to_string(index=False))



TASK 12: OUTLIERS (IQR RULE)
Q1: 58.6  Q3: 105.2  IQR: 46.6
A value is an outlier if it is below -11.3 or above 175.1
Number of outliers: 7
      Date       City  PM25
2025-04-15 Chandigarh 188.6
2025-10-22    Lucknow 179.6
2026-02-20     Jaipur 186.5
2026-04-24      Delhi 183.9
2026-07-22      Delhi 192.3
2026-09-13 Chandigarh 183.9
2027-07-02      Delhi 192.3


In [20]:
# TASK 13: Temperature and PM2.5 correlation
# ---------------------------------------------------------------------------
print("\n\nTASK 13: TEMPERATURE vs PM2.5 CORRELATION")
# Correlation goes from -1 to +1. Near 0 means almost no straight-line link.
temp_corr = df["Temperature_C"].corr(df["PM25"])
print("Correlation:", round(temp_corr, 3))

if abs(temp_corr) < 0.1:
    temp_strength = "negligible"
elif abs(temp_corr) < 0.3:
    temp_strength = "weak"
elif abs(temp_corr) < 0.5:
    temp_strength = "moderate"
else:
    temp_strength = "strong"
print("This is a", temp_strength, "relationship.")
print("(Correlation does not prove one thing causes the other.)")

humidity_corr = df["Humidity_Percent"].corr(df["PM25"])   # extra check



TASK 13: TEMPERATURE vs PM2.5 CORRELATION
Correlation: -0.021
This is a negligible relationship.
(Correlation does not prove one thing causes the other.)


In [21]:
# TASK 14: Share of readings above our chosen level
# ---------------------------------------------------------------------------
print("\n\nTASK 14: READINGS ABOVE", HIGH_LEVEL)
df["Above_Level"] = df["PM25"] > HIGH_LEVEL     # True / False for each row
percent_above = df["Above_Level"].mean() * 100  # True counts as 1, False as 0
print("Overall:", round(percent_above, 1), "% of readings")

print("\nBy city (%):")
percent_by_city = df.groupby("City")["Above_Level"].mean() * 100
print(percent_by_city.round(1).sort_values(ascending=False))

worst_city_percent = percent_by_city[worst_city]




TASK 14: READINGS ABOVE 60
Overall: 74.6 % of readings

By city (%):
City
Chandigarh    78.9
Lucknow       74.0
Jaipur        73.4
Delhi         72.2
Name: Above_Level, dtype: float64


In [22]:
# TASK 15: Five recommendations (built from the results above)
# ---------------------------------------------------------------------------
print("\n\nTASK 15: FIVE PUBLIC-AWARENESS RECOMMENDATIONS")

rec1 = (f"1. Focus on {worst_city}: it has the highest average PM2.5 "
        f"({worst_city_avg:.1f}) and {worst_city_percent:.1f}% of its readings are "
        f"above {HIGH_LEVEL}. Put extra monitors and alert boards there first.")

if spread_percent >= 15:
    rec2 = (f"2. Warn people before month {highest_month}, the month with the "
            f"highest average PM2.5.")
else:
    rec2 = (f"2. Run awareness campaigns all year: the monthly averages differ by "
            f"only {spread_percent:.0f}%, so no month is clearly safer. "
            f"Month {highest_month} is the highest.")

rec3 = (f"3. Protect sensitive people (children, older adults, people with asthma): "
        f"{bad_percent:.1f}% of readings are Poor or worse. On those days, "
        f"limit outdoor exercise and wear a well-fitted mask outside.")

rec4 = (f"4. Control dust as well as fine smoke: average PM10 is "
        f"{df['PM10'].mean():.1f} against {average_pm25:.1f} for PM2.5, and "
        f"{highest_pm10_city} has the highest PM10. Cover or water loose soil "
        f"on roads and building sites.")

rec5 = (f"5. Use live air-quality readings, not weather, to warn people: "
        f"temperature (r = {temp_corr:.2f}) and humidity (r = {humidity_corr:.2f}) "
        f"hardly predict PM2.5, and sudden spikes happen (highest: {max_pm25:.1f} "
        f"in {peak_row['City']} on {peak_row['Date'].date()}; "
        f"{len(outliers)} outliers in total).")

recommendations = [rec1, rec2, rec3, rec4, rec5]
for rec in recommendations:
    print("\n" + rec)



TASK 15: FIVE PUBLIC-AWARENESS RECOMMENDATIONS

1. Focus on Chandigarh: it has the highest average PM2.5 (85.8) and 78.9% of its readings are above 60. Put extra monitors and alert boards there first.

2. Run awareness campaigns all year: the monthly averages differ by only 4%, so no month is clearly safer. Month 1 is the highest.

3. Protect sensitive people (children, older adults, people with asthma): 39.7% of readings are Poor or worse. On those days, limit outdoor exercise and wear a well-fitted mask outside.

4. Control dust as well as fine smoke: average PM10 is 145.5 against 82.4 for PM2.5, and Chandigarh has the highest PM10. Cover or water loose soil on roads and building sites.

5. Use live air-quality readings, not weather, to warn people: temperature (r = -0.02) and humidity (r = 0.02) hardly predict PM2.5, and sudden spikes happen (highest: 192.3 in Delhi on 2026-07-22; 7 outliers in total).


In [23]:
# SAVE THE MAIN RESULTS TO FILES
# ---------------------------------------------------------------------------
city_ranking.to_csv("result_city_ranking.csv")
monthly_table.to_csv("result_monthly_pm25_by_city.csv")
top_dates.to_csv("result_high_pm25_dates.csv", index=False)
outliers[["Date", "City", "PM25"]].to_csv("result_outliers.csv", index=False)
df.to_csv("result_cleaned_data.csv", index=False)

with open("result_recommendations.txt", "w") as file:
    for rec in recommendations:
        file.write(rec + "\n\n")

print("\n\nDone! Result files were saved in this folder (names start with result_).")



Done! Result files were saved in this folder (names start with result_).
